# 06 - Event Study (`event_study.csv`)

## Purpose
Measure how each stock's price reacts to **earnings announcements** and **SEC 8-K filings** (supports **Q7**: how corporate events relate to stock price movements).

- **Part 1:** build the event reaction table - stock returns around each event and the reaction-day volume ratio.
- **Part 2:** add abnormal returns by subtracting the SPY (overall market) return.

## Inputs (`data/processed/`)
- `events_clean.csv` - earnings + 8-K events with their `reaction_date`
- `market_clean.csv` - daily prices, volume and returns (10 tickers x 1,254 trading days)
- `benchmark_prices.csv` - SPY daily close and return (Part 2)

## Output
`data/processed/event_study.csv` - one row per event.

| Column | Description |
|---|---|
| `event_id` | Event key from `events_clean.csv` |
| `ticker` | Stock ticker |
| `event_type` | `earnings` or `8-K` |
| `reaction_date` | First trading day the market could react (Day 0) |
| `ret_m1` | Stock return on Day -1 (previous trading day) |
| `ret_d0` | Stock return on Day 0 (reaction day) |
| `ret_p1` | Stock return on Day +1 (next trading day) |
| `ret_p2` | Stock return on Day +2 (second following trading day) |
| `volume_ratio_d0` | Day 0 volume / average volume of the previous 20 trading days |
| `surprise_pct` | EPS surprise % (earnings only, empty for 8-K) |
| `abn_d0` | Day 0 abnormal return = `ret_d0 - SPY_return_d0` (Part 2) |
| `car_m1_p1` | Cumulative abnormal return Day -1 to Day +1 = `AR(-1) + AR(0) + AR(+1)` (Part 2) |

## Method notes
- Returns are the daily close-to-close returns already stored in `market_clean.csv` (`return` = close_t / close_t-1 - 1).
- Days -1, +1, +2 are counted in **trading days** from `market_clean.csv`, not calendar days.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROCESSED_DIR = Path("../../data/processed")

EVENTS_PATH = PROCESSED_DIR / "events_clean.csv"
MARKET_PATH = PROCESSED_DIR / "market_clean.csv"
BENCHMARK_PATH = PROCESSED_DIR / "benchmark_prices.csv"
OUTPUT_PATH = PROCESSED_DIR / "event_study.csv"

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)

---
# Part 1 - Event Reaction Table

## Step 1 - Load `events_clean.csv` and `market_clean.csv`

In [2]:
events = pd.read_csv(EVENTS_PATH, parse_dates=["event_date", "reaction_date"])
market = pd.read_csv(MARKET_PATH, parse_dates=["date"])

market = market.sort_values(["ticker", "date"]).reset_index(drop=True)

print(f"events_clean : {events.shape[0]:,} rows x {events.shape[1]} cols")
print(events["event_type"].value_counts().to_string())
print()
print(f"market_clean : {market.shape[0]:,} rows, {market['ticker'].nunique()} tickers, "
      f"{market['date'].nunique():,} trading days ({market['date'].min().date()} -> {market['date'].max().date()})")

events_clean : 844 rows x 12 cols
event_type
8-K         644
earnings    200

market_clean : 12,540 rows, 10 tickers, 1,254 trading days (2021-01-04 -> 2025-12-30)


## Step 2 - Remove 8-K records whose `items` contain `2.02`
Item 2.02 (*Results of operations*) is the 8-K that companies file for their earnings release, so it duplicates the earnings announcement already in the table.

In [3]:
is_8k_202 = (events["event_type"] == "8-K") & events["items"].str.contains("2.02", regex=False, na=False)

print(f"8-K records with item 2.02 removed : {is_8k_202.sum()}")
events_f = events.loc[~is_8k_202].copy()

print(f"Events remaining                    : {len(events_f)}")
print(events_f["event_type"].value_counts().to_string())

8-K records with item 2.02 removed : 227
Events remaining                    : 617
event_type
8-K         417
earnings    200


## Steps 3-5 - Returns on Day -1, 0, +1, +2 and `volume_ratio_d0`
For each ticker, the trading days in `market_clean.csv` are in date order, so shifting by one row moves exactly one **trading day**:

- `ret_m1` = return of the previous row (Day -1)
- `ret_d0` = return of the reaction-day row (Day 0)
- `ret_p1`, `ret_p2` = return of the next / second next row (Day +1, Day +2)
- `volume_ratio_d0` = Day 0 volume / mean volume of the **20 trading days before** Day 0 (Day 0 itself is not included). It is left empty if fewer than 20 prior trading days exist.

In [4]:
g = market.groupby("ticker", group_keys=False)

window = market[["ticker", "date"]].copy()
window["ret_m1"] = g["return"].shift(1)
window["ret_d0"] = market["return"]
window["ret_p1"] = g["return"].shift(-1)
window["ret_p2"] = g["return"].shift(-2)

avg_vol_prev20 = g["volume"].transform(lambda v: v.shift(1).rolling(20, min_periods=20).mean())
window["volume_ratio_d0"] = market["volume"] / avg_vol_prev20

window.head(3)

,ticker,date,ret_m1,ret_d0,ret_p1,ret_p2,volume_ratio_d0
0,AAPL,2021-01-04,NaN,NaN,0.012363,-0.033661,NaN
1,AAPL,2021-01-05,NaN,0.012363,-0.033661,0.034124,NaN
2,AAPL,2021-01-06,0.012363,-0.033661,0.034124,0.008631,NaN


## Join every event to its reaction day
Every `reaction_date` must exist as a trading day for that ticker in `market_clean.csv`.

In [5]:
event_study = events_f.merge(
    window,
    left_on=["ticker", "reaction_date"],
    right_on=["ticker", "date"],
    how="left",
    validate="many_to_one",
    indicator=True,
)

unmatched = (event_study["_merge"] != "both").sum()
print(f"Events without a matching trading day: {unmatched}")
assert unmatched == 0

Events without a matching trading day: 0


## Step 6 - Save `event_study.csv`

In [6]:
PART1_COLUMNS = [
    "event_id", "ticker", "event_type", "reaction_date",
    "ret_m1", "ret_d0", "ret_p1", "ret_p2",
    "volume_ratio_d0", "surprise_pct",
]

event_study = (
    event_study[PART1_COLUMNS]
    .sort_values("event_id")
    .reset_index(drop=True)
)
event_study[["ret_m1", "ret_d0", "ret_p1", "ret_p2", "volume_ratio_d0"]] = (
    event_study[["ret_m1", "ret_d0", "ret_p1", "ret_p2", "volume_ratio_d0"]].round(6)
)

event_study.to_csv(OUTPUT_PATH, index=False, date_format="%Y-%m-%d")
print(f"Saved {OUTPUT_PATH} ({len(event_study)} rows x {event_study.shape[1]} cols)")
event_study.head()

Saved ..\..\data\processed\event_study.csv (617 rows x 10 cols)


,event_id,ticker,event_type,reaction_date,ret_m1,ret_d0,ret_p1,ret_p2,volume_ratio_d0,surprise_pct
0,2,TSLA,8-K,2021-01-04,NaN,NaN,0.007317,0.028390,NaN,NaN
1,3,AAPL,8-K,2021-01-05,NaN,0.012363,-0.033661,0.034124,NaN,NaN
2,4,LLY,8-K,2021-01-14,0.037472,0.006415,0.021854,0.040415,NaN,NaN
3,6,JPM,8-K,2021-01-15,0.005842,-0.017921,-0.004328,-0.014995,NaN,NaN
4,7,JPM,earnings,2021-01-15,0.005842,-0.017921,-0.004328,-0.014995,NaN,15.81


## Completion checks (Part 1)
1. One row per event.
2. AMZN on 2024-08-02 has a reaction-day return of about **-8.8%**.

In [7]:
# 1. One row per event
assert event_study["event_id"].is_unique
assert len(event_study) == len(events_f)
print(f"One row per event: OK ({len(event_study)} rows, {event_study['event_id'].nunique()} unique event_id)")

# 2. AMZN 2024-08-02
amzn = event_study[(event_study["ticker"] == "AMZN") & (event_study["reaction_date"] == "2024-08-02")]
print()
print(amzn.to_string(index=False))
assert len(amzn) == 1 and abs(amzn["ret_d0"].iloc[0] - (-0.088)) < 0.001
print(f"\nAMZN 2024-08-02 ret_d0 = {amzn['ret_d0'].iloc[0]:.2%}  -> OK (~ -8.8%)")

One row per event: OK (617 rows, 617 unique event_id)

 event_id ticker event_type reaction_date    ret_m1    ret_d0    ret_p1   ret_p2  volume_ratio_d0  surprise_pct
      631   AMZN   earnings    2024-08-02 -0.015563 -0.087847 -0.040977 0.005651         3.449512         23.76

AMZN 2024-08-02 ret_d0 = -8.78%  -> OK (~ -8.8%)


### Missing values
Empty values are expected only at the start of the price history: an event on the first trading day (2021-01-04) has no Day -1 or Day 0 return, and events within the first 20 trading days have no 20-day volume average. `surprise_pct` is empty for all 8-K events by design.

In [8]:
print(event_study.isna().sum().to_string())
print()
print("Events with empty return / volume ratio columns:")
print(event_study[event_study[["ret_m1", "ret_d0", "ret_p1", "ret_p2", "volume_ratio_d0"]].isna().any(axis=1)]
      [["event_id", "ticker", "event_type", "reaction_date", "ret_m1", "ret_d0", "volume_ratio_d0"]].to_string(index=False))

first_20_days = np.sort(market["date"].unique())[19]
assert (event_study.loc[event_study["volume_ratio_d0"].isna(), "reaction_date"] <= first_20_days).all()
assert event_study.loc[event_study["event_type"] == "earnings", "surprise_pct"].notna().all()

event_id             0
ticker               0
event_type           0
reaction_date        0
ret_m1               2
ret_d0               1
ret_p1               0
ret_p2               0
volume_ratio_d0     12
surprise_pct       417

Events with empty return / volume ratio columns:
 event_id ticker event_type reaction_date    ret_m1    ret_d0  volume_ratio_d0
        2   TSLA        8-K    2021-01-04       NaN       NaN              NaN
        3   AAPL        8-K    2021-01-05       NaN  0.012363              NaN
        4    LLY        8-K    2021-01-14  0.037472  0.006415              NaN
        6    JPM        8-K    2021-01-15  0.005842 -0.017921              NaN
        7    JPM   earnings    2021-01-15  0.005842 -0.017921              NaN
        8    JPM        8-K    2021-01-21 -0.014995 -0.008384              NaN
       10    JNJ   earnings    2021-01-26  0.014858  0.027112              NaN
       12   MSFT   earnings    2021-01-27  0.012199  0.002453              NaN
       14

---
# Part 2 - Abnormal Returns

## Step 1 - Load `event_study.csv` and `benchmark_prices.csv`

In [9]:
event_study = pd.read_csv(OUTPUT_PATH, parse_dates=["reaction_date"])
spy = pd.read_csv(BENCHMARK_PATH, parse_dates=["date"]).sort_values("date").reset_index(drop=True)

print(f"event_study : {event_study.shape[0]} rows")
print(f"benchmark   : {spy.shape[0]:,} rows, symbol = {spy['symbol'].unique().tolist()}")

# SPY must cover exactly the same trading days as market_clean
same_calendar = np.array_equal(np.sort(spy["date"].unique()), np.sort(market["date"].unique()))
print(f"SPY dates identical to market_clean trading days: {same_calendar}")
assert same_calendar

event_study : 617 rows
benchmark   : 1,254 rows, symbol = ['SPY']
SPY dates identical to market_clean trading days: True


## Step 2 - Match each event's reaction date with the SPY return
Because SPY and the stocks share the same trading calendar, SPY's Day -1 / Day 0 / Day +1 returns are taken the same way as the stock returns (one row = one trading day).

In [10]:
spy_window = pd.DataFrame({
    "reaction_date": spy["date"],
    "spy_m1": spy["return"].shift(1),
    "spy_d0": spy["return"],
    "spy_p1": spy["return"].shift(-1),
})

es = event_study.merge(spy_window, on="reaction_date", how="left", validate="many_to_one")
assert len(es) == len(event_study)
print(f"Events without a SPY reaction-day return: {es['spy_d0'].isna().sum()}")

Events without a SPY reaction-day return: 1


## Steps 3-4 - Abnormal returns
$$abn\_d0 = ret\_d0 - SPY\_return\_d0$$

$$CAR_{-1,+1} = AR_{-1} + AR_{0} + AR_{+1}, \quad AR_t = ret_t - SPY\_return_t$$

In [11]:
ar_m1 = es["ret_m1"] - es["spy_m1"]
ar_d0 = es["ret_d0"] - es["spy_d0"]
ar_p1 = es["ret_p1"] - es["spy_p1"]

es["abn_d0"] = ar_d0.round(6)
es["car_m1_p1"] = (ar_m1 + ar_d0 + ar_p1).round(6)

es[["event_id", "ticker", "event_type", "reaction_date", "ret_d0", "spy_d0", "abn_d0", "car_m1_p1"]].head()

,event_id,ticker,event_type,reaction_date,ret_d0,spy_d0,abn_d0,car_m1_p1
0,2,TSLA,8-K,2021-01-04,NaN,NaN,NaN,NaN
1,3,AAPL,8-K,2021-01-05,0.012363,0.006887,0.005476,NaN
2,4,LLY,8-K,2021-01-14,0.006415,-0.003502,0.009917,0.073843
3,6,JPM,8-K,2021-01-15,-0.017921,-0.007293,-0.010628,-0.013464
4,7,JPM,earnings,2021-01-15,-0.017921,-0.007293,-0.010628,-0.013464


## Step 5 - Add `abn_d0` and `car_m1_p1` to `event_study.csv`

In [12]:
FINAL_COLUMNS = PART1_COLUMNS + ["abn_d0", "car_m1_p1"]

event_study = es[FINAL_COLUMNS]
assert event_study["event_id"].is_unique

event_study.to_csv(OUTPUT_PATH, index=False, date_format="%Y-%m-%d")
print(f"Saved {OUTPUT_PATH} ({len(event_study)} rows x {event_study.shape[1]} cols)")
print(event_study[["abn_d0", "car_m1_p1"]].isna().sum().to_string())

Saved ..\..\data\processed\event_study.csv (617 rows x 12 cols)
abn_d0       1
car_m1_p1    2


## Completion check (Part 2) - Manual verification of three rows
For three events (including AMZN on 2024-08-02) the returns are recomputed **by hand from the close prices** (`close_t / close_t-1 - 1`) of the stock and SPY, independently of the stored `return` columns, and compared with the values in `event_study.csv`.

| event_id | Ticker | Type | Reaction date |
|---|---|---|---|
| 631 | AMZN | earnings | 2024-08-02 |
| 99 | NVDA | earnings | 2021-05-27 |
| 40 | MSFT | 8-K | 2021-03-02 |

In [13]:
CHECK_EVENTS = [631, 99, 40]
trading_days = np.sort(market["date"].unique())
stock_close = market.set_index(["ticker", "date"])["close"]
spy_close = spy.set_index("date")["close"]

for eid in CHECK_EVENTS:
    row = event_study.loc[event_study["event_id"] == eid].iloc[0]
    t, d0 = row["ticker"], row["reaction_date"]
    i = np.searchsorted(trading_days, d0)
    d = {k: trading_days[i + k] for k in (-2, -1, 0, 1)}   # Day -2 is needed for the Day -1 return

    def r(close, k, key=None):
        cur = close[(key, d[k])] if key else close[d[k]]
        prev = close[(key, d[k - 1])] if key else close[d[k - 1]]
        return cur / prev - 1

    ar = {k: r(stock_close, k, t) - r(spy_close, k) for k in (-1, 0, 1)}
    manual_abn_d0 = ar[0]
    manual_car = ar[-1] + ar[0] + ar[1]

    print(f"event_id {eid} | {t} | {row['event_type']} | Day 0 = {pd.Timestamp(d0).date()}")
    for k in (-1, 0, 1):
        print(f"   Day {k:+d} ({pd.Timestamp(d[k]).date()}): stock {r(stock_close, k, t):+.4%}  SPY {r(spy_close, k):+.4%}  AR {ar[k]:+.4%}")
    print(f"   abn_d0    manual {manual_abn_d0:+.6f} | file {row['abn_d0']:+.6f}")
    print(f"   car_m1_p1 manual {manual_car:+.6f} | file {row['car_m1_p1']:+.6f}")
    assert abs(manual_abn_d0 - row["abn_d0"]) < 1e-5 and abs(manual_car - row["car_m1_p1"]) < 1e-5
    print("   -> matches\n")

event_id 631 | AMZN | earnings | Day 0 = 2024-08-02
   Day -1 (2024-08-01): stock -1.5563%  SPY -1.4161%  AR -0.1402%
   Day +0 (2024-08-02): stock -8.7847%  SPY -1.8618%  AR -6.9229%
   Day +1 (2024-08-05): stock -4.0977%  SPY -2.9124%  AR -1.1853%
   abn_d0    manual -0.069229 | file -0.069229
   car_m1_p1 manual -0.082484 | file -0.082484
   -> matches

event_id 99 | NVDA | earnings | Day 0 = 2021-05-27
   Day -1 (2021-05-26): stock +0.3339%  SPY +0.1985%  AR +0.1354%
   Day +0 (2021-05-27): stock -1.3504%  SPY +0.0525%  AR -1.4029%
   Day +1 (2021-05-28): stock +4.8846%  SPY +0.1789%  AR +4.7057%
   abn_d0    manual -0.014029 | file -0.014028
   car_m1_p1 manual +0.034382 | file +0.034381
   -> matches

event_id 40 | MSFT | 8-K | Day 0 = 2021-03-02
   Day -1 (2021-03-01): stock +1.9623%  SPY +2.4240%  AR -0.4617%
   Day +0 (2021-03-02): stock -1.2957%  SPY -0.7803%  AR -0.5154%
   Day +1 (2021-03-03): stock -2.6981%  SPY -1.3246%  AR -1.3735%
   abn_d0    manual -0.005154 | file -0

## Final output preview

In [14]:
print(f"{len(event_study)} events | columns: {list(event_study.columns)}")
event_study.head(10)

617 events | columns: ['event_id', 'ticker', 'event_type', 'reaction_date', 'ret_m1', 'ret_d0', 'ret_p1', 'ret_p2', 'volume_ratio_d0', 'surprise_pct', 'abn_d0', 'car_m1_p1']


,event_id,ticker,event_type,reaction_date,ret_m1,ret_d0,ret_p1,ret_p2,volume_ratio_d0,surprise_pct,abn_d0,car_m1_p1
0,2,TSLA,8-K,2021-01-04,NaN,NaN,0.007317,0.028390,NaN,NaN,NaN,NaN
1,3,AAPL,8-K,2021-01-05,NaN,0.012363,-0.033661,0.034124,NaN,NaN,0.005476,NaN
2,4,LLY,8-K,2021-01-14,0.037472,0.006415,0.021854,0.040415,NaN,NaN,0.009917,0.073843
3,6,JPM,8-K,2021-01-15,0.005842,-0.017921,-0.004328,-0.014995,NaN,NaN,-0.010628,-0.013464
4,7,JPM,earnings,2021-01-15,0.005842,-0.017921,-0.004328,-0.014995,NaN,15.81,-0.010628,-0.013464
5,8,JPM,8-K,2021-01-21,-0.014995,-0.008384,-0.007713,-0.012482,NaN,NaN,-0.009296,-0.042304
6,10,JNJ,earnings,2021-01-26,0.014858,0.027112,-0.015251,0.007624,NaN,1.76,0.028673,0.048776
7,12,MSFT,earnings,2021-01-27,0.012199,0.002453,0.025891,-0.029171,NaN,23.99,0.026893,0.057944
8,14,AAPL,earnings,2021-01-28,-0.007684,-0.034985,-0.037420,0.016520,NaN,18.67,-0.043585,-0.044229
9,15,AMZN,8-K,2021-01-27,0.009754,-0.028126,0.001559,-0.009705,NaN,NaN,-0.003686,0.000588
